# Figure 4: Behavioral Motif Encoding and Decoding

**Paper**: Parodi et al. (2026)

This notebook reproduces the motif analysis for Figure 4:
- **Panel A**: Motif decoding accuracy (logistic regression from neural population)
- **Panel B**: Electrode-level tuning and selectivity
- **Panel C**: Transition matrix between behavioral syllables
- **Panel D**: Peri-event time histogram (PETH) around motif transitions

**Behavioral syllables** are defined by an unsupervised motion segmentation pipeline (keypoint-MoSeq style) that identifies 10-12 recurring posture-movement patterns (e.g., Rest, Stand, Pace, Pivot, Inspect).

**Note**: This notebook runs with 10 permutations for speed (~4 min). Production uses 1000 permutations.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*lbfgs failed to converge.*")
warnings.filterwarnings("ignore", message=".*y_pred contains classes not in y_true.*")

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})
print("Setup complete.")

## 4.1 Run Motif Analysis

Run the full Fig 4 pipeline: motif decoding, tuning, transition analysis, and PETH.

In [ ]:
from freemat.single_fm.encoding.run_fig4 import Fig4Config, run_fig4_session
import tempfile

config = Fig4Config(
    bin_s=0.5,
    top_n=10,          # Top 10 motif classes
    n_perm=10,         # 10 permutations (production: 1000)
    debug=True,
)

with tempfile.TemporaryDirectory() as tmpdir:
    results = run_fig4_session(
        session_yaml=str(REPO_ROOT / "configs" / "session_M2_0526.yaml"),
        config=config,
        output_dir=tmpdir,
    )
    # Extract results
    decoding = results.get("decoding")
    tuning = results.get("tuning")
    transition_matrix = results.get("transition_matrix")
    peth = results.get("peth")
    fine_peth = results.get("fine_peth")

print(f"\nDecoding accuracy: {decoding['accuracy']:.1%} (chance: {decoding['chance']:.1%})")
print(f"Number of motif classes: {decoding['n_classes']}")
print(f"Transition matrix shape: {transition_matrix.shape}")
print(f"PETH shape: {peth.shape} (units × time bins)")

## 4.2 Panel A: Decoding Accuracy

Motif decoding accuracy from neural population activity using temporal cross-validation with 2s embargo.

In [ ]:
# Panel A: Decoding accuracy bar chart
fig, ax = plt.subplots(figsize=(6, 5))

acc = decoding["accuracy"]
chance = decoding["chance"]
n_classes = decoding["n_classes"]

ax.bar(["Neural\nDecoding", "Chance"], [acc, chance],
       color=["#1976D2", "#BDBDBD"], edgecolor="black", linewidth=0.5, width=0.5)
ax.set_ylabel("Balanced Accuracy")
ax.set_title(f"Fig 4A: Motif Decoding — {n_classes} classes")
ax.set_ylim(0, max(acc, chance) * 1.3)

# Annotate
ax.text(0, acc + 0.01, f"{acc:.1%}", ha="center", fontsize=12, fontweight="bold")
ax.text(1, chance + 0.01, f"{chance:.1%}", ha="center", fontsize=12)

plt.tight_layout()
plt.show()

print(f"Decoding: {acc:.1%} vs chance {chance:.1%} ({acc/chance:.1f}× above chance)")

## 4.3 Panel C: Transition Matrix

Probability of transitioning between motifs. High diagonal = strong self-transition (behavioral persistence).

In [ ]:
# Panel C: Transition matrix
fig, ax = plt.subplots(figsize=(8, 7))

im = ax.imshow(transition_matrix, cmap="Blues", aspect="auto", vmin=0, vmax=1)
ax.set_xlabel("Next Motif")
ax.set_ylabel("Current Motif")
ax.set_title("Fig 4C: Motif Transition Probabilities")

# Add text annotations
for i in range(transition_matrix.shape[0]):
    for j in range(transition_matrix.shape[1]):
        val = transition_matrix[i, j]
        if val > 0.01:
            color = "white" if val > 0.5 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, label="Transition Probability", shrink=0.8)

diag_mean = np.diag(transition_matrix).mean()
print(f"Mean self-transition probability: {diag_mean:.3f}")

plt.tight_layout()
plt.show()